# 推理服务与量化补充线 · 第 3/8 课：Paged KV、Block Table 与 Prefix Cache

> 状态：**未开始**  
> 逐课通过制：未完成代码、问答与边界解释前，不进入下一课。

## 本课目标与完成标准

本课产出：实现逻辑 token 到物理 KV block 的映射，并解释内部碎片、共享与隔离。

通过要求：唯一代码填空题通过给定检查；Q1～Q3 都给出因果链；能指出一个正确性边界和一个性能/运营取舍；总分至少 8/10。

## 与既有六条主线的边界

本课借用操作系统分页心智模型，但只讨论推理 KV 管理；不重复 `cuda/` 的实际地址计算。

前置：train 第 1～5 课、CUDA/Triton 基础、Transformer attention。本课只补 AI Infra 缺口，不重复已经完成的 kernel 或并行算法推导。

## 核心心智模型

### 它是什么、解决什么问题

Paged KV 将每个序列的逻辑 token 空间切成定长块，通过 block table 指向非连续物理块；prefix cache 让内容相同且配置兼容的前缀复用块。

### 数据与控制如何流动

attention kernel 先由逻辑 block 查物理 block，再用 token offset 定位。共享块需引用计数或 copy-on-write，cache key 必须包含 token 序列及影响 KV 的模型/适配器配置。

### 正确性条件与常见误区

block table 越界、已释放块复用、租户 key 冲突都会造成错误输出或数据泄漏。最后一个块的未用槽位是内部碎片，不应当成有效 token。

### 性能、成本与工程取舍

小 block 降碎片、提高命中粒度，但 block table 和调度 metadata 更多；大 block 元数据少，却浪费尾部容量并降低 prefix 命中边界精度。

## 具体演示

block_size=4，逻辑 block table=[9,2,7]：token 6 位于逻辑块 1、块内 offset 2，因此物理位置是 block 2 的 offset 2。

请先独立复述“输入 → 状态变化 → 输出/指标”，再开始填空。

## 实践任务：唯一代码填空题

补齐 paged KV 地址翻译，并拒绝负 token 或缺失逻辑块。

只能修改 `TODO`/`______` 位置，不得删除断言或放宽通过条件。

In [ ]:
def translate_token(block_table, token_index, block_size):
    if token_index < 0 or block_size <= 0:
        raise ValueError("invalid token or block size")
    logical_block, offset = divmod(token_index, block_size)
    if logical_block >= len(block_table):
        raise IndexError("token is not allocated")
    # TODO：返回物理 block id 与块内位置。
    return ______

assert translate_token([9, 2, 7], 6, 4) == (2, 2)
assert translate_token([9, 2, 7], 11, 4) == (7, 3)


### 检查方法

运行本单元格，所有断言必须通过；再补一个边界输入并解释预期。

### Q1

PagedAttention 为什么主要解决外部碎片和过度预留，而不是让 KV 本身变小？

**你的答案：**


### Q2

prefix cache 只用 token IDs 做 key 有什么错误或安全风险？

**你的答案：**


### Q3

block_size 从 16 改成 64，命中率、metadata 与碎片如何变化？

**你的答案：**


## 评分规则

- 代码 4 分：正常输入 2 分、边界输入 1 分、解释 1 分；
- Q1～Q3 各 2 分；
- 一票否决：混淆测量与推测、忽略租户/请求隔离、用平均值掩盖尾延迟、删除失败路径。

## 参考资料

- [PagedAttention / vLLM paper](https://arxiv.org/abs/2309.06180)
- [vLLM serving documentation](https://docs.vllm.ai/en/latest/cli/serve/)

API 与平台能力会演进；部署前应按目标版本重新核对。